# Watt The Hack Controller Training Starter

<a href="https://colab.research.google.com/github/ScrapMetal1/Watt-The-Hack/blob/master/notebooks/training_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome to the advanced track! In this notebook, you will learn how to write a controller for the Watt The Hack simulation engine. We will cover installing the engine, writing basic controllers, evaluating them against scenarios, and exporting an ML model's weights.

## 1. Install the Simulation Engine
First, let's install the Watt The Hack engine directly from GitHub.

In [ ]:
!pip install git+https://github.com/AaronEliasZachariah/GridLock-Engine.git@main

## 2. Setting up the Simulation
We can instantiate the `NetworkEngine` and run it for a few steps with a simple "do-nothing" controller.

In [ ]:
from watt_the_hack.engine.networkengine import NetworkEngine
from watt_the_hack.data_loaders.default_profiles import default_initial_state

engine = NetworkEngine()

def do_nothing_controller(state):
    return {
        "battery_flow_kw": 0.0,
        "emergency_generator": 0.0,
        "curtail_solar": 0.0,
        "fcas_reserve_kw": 0.0,
    }

state = default_initial_state(steps=24 * 4)
state = engine.add_forecast_to_state(state)

for step in range(10):
    action = do_nothing_controller(state)
    state, outputs = engine.step(state, action)
    print(f"Step {step}: Cost = ${outputs['step_cost']:.2f}, Battery SOC = {state['soc']:.2f}")

## 3. A Rule-Based Controller
A slightly more advanced controller that charges when prices are low and discharges when prices are high.

In [ ]:
def rule_based_controller(state):
    action = {
        "battery_flow_kw": 0.0,
        "emergency_generator": 0.0,
        "curtail_solar": 0.0,
        "fcas_reserve_kw": 0.0,
    }
    
    current_price = state["price"]
    soc = state["soc"]
    
    # Simple arbitrage logic
    if current_price < 0.10 and soc < 0.9:
        action["battery_flow_kw"] = -20.0  # Charge
    elif current_price > 0.30 and soc > 0.1:
        action["battery_flow_kw"] = 20.0   # Discharge
        
    # Hold some FCAS if we aren't doing anything else
    if action["battery_flow_kw"] == 0.0:
        action["fcas_reserve_kw"] = 10.0
        
    return action

state = default_initial_state(steps=24 * 4)
state = engine.add_forecast_to_state(state)
total_cost = 0
for _ in range(96):
    action = rule_based_controller(state)
    state, outputs = engine.step(state, action)
    total_cost += outputs["step_cost"]

print(f"Rule-based total cost: ${total_cost:.2f}")

## 4. Reinforcement Learning Scaffold
You can wrap the engine in a gym environment. Here is a simple scaffold.

In [ ]:
import numpy as np

class SimpleModel:
    """A dummy model to demonstrate weight extraction."""
    def __init__(self):
        # Random weights for illustration: 4 actions based on 3 state variables
        self.weights = np.random.randn(4, 3)
        
    def predict(self, obs):
        return np.dot(self.weights, obs)

model = SimpleModel()

## 5. Compiling Weights to Literals
Since the sandbox restricts imports (no `numpy` or `torch`), you must bake your model's weights directly into your submission code as Python literals.

In [ ]:
def export_weights_to_code(model, filename="submission.py"):
    weights_list = model.weights.tolist()
    
    code = f"""# Auto-generated submission
WEIGHTS = {weights_list}

def controller(state):
    # Map state to vector
    obs = [state.get("soc", 0), state.get("price", 0), state.get("demand", 0)]
    
    # Manual dot product
    actions = [sum(w * o for w, o in zip(weight_row, obs)) for weight_row in WEIGHTS]
    
    # Action clamping and conversion
    return {{
        \"battery_flow_kw\": max(-50.0, min(50.0, actions[0])),
        \"emergency_generator\": max(0.0, min(50.0, actions[1])),
        \"curtail_solar\": max(0.0, min(state.get(\"solar\", 0.0), actions[2])),
        \"fcas_reserve_kw\": max(0.0, min(50.0, actions[3])),
    }}
"""
    with open(filename, "w") as f:
        f.write(code)
    print(f"Exported controller to {filename}")

export_weights_to_code(model)

# Let's view the generated code:
with open("submission.py", "r") as f:
    print(f.read())

## 6. Submitting to the Cloud Leaderboard

When you're ready to score your controller against the hidden judging scenarios, submit it to the **cloud evaluation API**. This is different from the quick `/sim/run` playground — your code runs in an isolated Kubernetes container, gets a final score, and lands on the leaderboard.

**The flow is asynchronous:**
1. POST your ZIP to `/submissions` → get a `submission_id` back immediately
2. Poll `/submissions/{id}` until status is `COMPLETED`
3. GET `/submissions/{id}/score` to retrieve your scores

**Your ZIP must contain:**
- `strategy.py` — defines `class Strategy` with `.step(state)` (and optionally `.plan()` / `.replan()` for agentic scenarios)
- `requirements.txt` — your pip dependencies (`openai`, `anthropic` available for agentic scenarios)
- `metadata.json` — `{"entrypoint": "strategy.py", "class_name": "Strategy"}`

**You need:**
- `API_URL`: the gateway URL provided by organisers (e.g. `https://api.watt-the-hack.example.com`)
- `TEAM_TOKEN`: your team's `X-Team-Token` issued by organisers

In [ ]:
import io
import json
import time
import zipfile
import requests

# ── CONFIG — set these for your team ───────────────────────────────────────
API_URL    = "https://eval.YOUR-DOMAIN.com"   # ← provided by organisers
TEAM_TOKEN = "your-team-token-here"           # ← provided by organisers

# ── 1. Write your strategy (class-based for cloud submission) ─────────────
STRATEGY_CODE = '''
class Strategy:
    def __init__(self):
        self.last_price = None

    def step(self, state):
        action = {
            "battery_flow_kw": 0.0,
            "emergency_generator": 0.0,
            "curtail_solar": 0.0,
            "fcas_reserve_kw": 0.0,
        }
        price = state.get("price", 0.0)
        soc   = state.get("soc", 0.0)
        if price < 0.10 and soc < 0.9:
            action["battery_flow_kw"] = -20.0
        elif price > 0.30 and soc > 0.1:
            action["battery_flow_kw"] = 20.0
        return action
'''

REQUIREMENTS = ""  # add e.g. "openai==1.35.0
" if using LLM
METADATA = {"entrypoint": "strategy.py", "class_name": "Strategy"}

# ── 2. Package as ZIP in memory ───────────────────────────────────────────
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("strategy.py",      STRATEGY_CODE)
    zf.writestr("requirements.txt", REQUIREMENTS)
    zf.writestr("metadata.json",    json.dumps(METADATA))
buf.seek(0)

# ── 3. Upload submission ─────────────────────────────────────────────────
print("Uploading submission...")
resp = requests.post(
    f"{API_URL}/submissions",
    headers={"X-Team-Token": TEAM_TOKEN},
    files={"file": ("submission.zip", buf, "application/zip")},
    data={"llm_required": "false"},  # set "true" if your Strategy uses LLM
)
if resp.status_code != 202:
    raise SystemExit(f"Upload failed ({resp.status_code}): {resp.text}")

submission_id = resp.json()["submission_id"]
print(f"✓ Submitted. ID: {submission_id}")

# ── 4. Poll until evaluation completes ───────────────────────────────────
print("Waiting for evaluation (build + run, typically 2–5 min)...")
while True:
    s = requests.get(
        f"{API_URL}/submissions/{submission_id}",
        headers={"X-Team-Token": TEAM_TOKEN},
    ).json()
    status = s["status"]
    print(f"  [{status}] {s.get('status_message') or ''}")
    if status in ("COMPLETED", "EVAL_FAILED", "BUILD_FAILED", "TIMEOUT"):
        break
    time.sleep(10)

# ── 5. Fetch the score ──────────────────────────────────────────────────
if status == "COMPLETED":
    score = requests.get(
        f"{API_URL}/submissions/{submission_id}/score",
        headers={"X-Team-Token": TEAM_TOKEN},
    ).json()
    print("
✅ Submission scored!")
    print(json.dumps(score, indent=2))
else:
    print(f"
❌ Submission ended with status: {status}")
    logs = requests.get(
        f"{API_URL}/submissions/{submission_id}/logs",
        headers={"X-Team-Token": TEAM_TOKEN},
    ).text
    print("Last 2000 chars of logs:
", logs[-2000:])

## 7. Agentic Hybrid Controller (OpenAI)

For agentic scenarios, you can define a `Strategy` class with `.plan()` and `.replan()` methods to use an LLM. The LLM can read the scenario briefing and set a high-level strategy (like a target SOC), which the fast `.step()` method then executes.

<script src="https://gist.github.com/AaronEliasZachariah/67848957865aaabd5dfecffe3b5ad3a2.js"></script>